# Construyendo tu propio algoritmo de entrenamiento dentro de un contenedor (BYOC) de Amazon SageMaker


### Construyendo y registrando el container

El siguiente código shell muestra cómo construir el container imagen usando `docker build` y subirlo a ECR usando `docker push`. Este código también está disponible como el shell script `container/build-and-push.sh`, que puedes correr como `build-and-push.sh decision_trees` para construir el image `sales_supermarket`.

Este código busca un repositorio ECR en la cuenta que estás usando y la región por defecto actual (si estás usando una SageMaker notebook instance, esta será la región donde se creó la notebook instance). Si el repositorio no existe, el script lo creará.

In [3]:
# S3 prefix
prefix = "sales_supermarket"

# Define IAM role
import boto3
import re

import os
import numpy as np
import pandas as pd
from sagemaker import get_execution_role

role = get_execution_role()

## Crear la sesión

La sesión recuerda los parámetros de conexión a SageMaker. La usaremos para realizar todas nuestras operaciones de SageMaker.

In [4]:
import sagemaker as sage
from time import gmtime, strftime

sess = sage.Session()

## Subir los datos de entrenamiento

Cuando entrenas modelos grandes con grandes cantidades de datos, normalmente usarás herramientas de big data, como Amazon Athena, AWS Glue, o Amazon EMR, para crear tus datos en S3. Para este ejemplo, usamos el clásico [Iris dataset](https://en.wikipedia.org/wiki/Iris_flower_data_set), que hemos incluido.

Podemos usar las herramientas del SageMaker Python SDK para subir los datos a un bucket por defecto.

In [5]:
WORK_DIRECTORY = "data"

default_bucket = sess.default_bucket()

default_bucket_prefix = sess.default_bucket_prefix

# If a default bucket prefix is specified, append it to the s3 path
if default_bucket_prefix:
    prefix = f"{default_bucket_prefix}/{prefix}"

data_location = sess.upload_data(WORK_DIRECTORY, bucket=default_bucket, key_prefix=prefix)

## Crear un estimator y entrenar el modelo

Para usar SageMaker y entrenar nuestro algoritmo, crearemos un `Estimator` que define cómo usar el container para entrenar. Esto incluye la configuración necesaria para invocar el training de SageMaker:

* El __container name__. Se construye como en los comandos shell anteriores.
* El __role__. Como se definió arriba.
* El __instance count__ que es el número de máquinas a usar para training.
* El __instance type__ que es el tipo de máquina a usar para training.
* El __output path__ determina dónde se escribirá el model artifact.
* La __session__ es el objeto SageMaker session que definimos arriba.

Luego usamos fit() en el estimator para entrenar con los datos que subimos anteriormente.

In [6]:
account = sess.boto_session.client("sts").get_caller_identity()["Account"]
region = sess.boto_session.region_name
image = "{}.dkr.ecr.{}.amazonaws.com/supermarket:latest".format(account, region)
s3_output_path = "s3://{}/output".format(default_bucket)

# If a default bucket prefix is specified, append it to the s3 path
if default_bucket_prefix:
    s3_output_path = "s3://{}/{}/output".format(default_bucket, default_bucket_prefix)

tree = sage.estimator.Estimator(
    image,
    role,
    1,
    "ml.c4.2xlarge",
    output_path=s3_output_path,
    sagemaker_session=sess,
)

tree.fit(data_location)

INFO:sagemaker:Creating training-job with name: supermarket-2026-03-08-20-22-58-723


2026-03-08 20:23:00 Starting - Starting the training job...
2026-03-08 20:23:15 Starting - Preparing the instances for training...
2026-03-08 20:23:57 Downloading - Downloading the training image
2026-03-08 20:23:57 Training - Training image download completed. Training in progress...2026-03-08 20:24:08,249 - __main__ - INFO - Iniciando Carga de Datos...
2026-03-08 20:24:08,614 - __main__ - INFO - Dividiendo el set de para entrenamiento y validación...
2026-03-08 20:24:10,559 - __main__ - INFO - Iniciando búsqueda de hiperparámetros...
Fitting 3 folds for each of 10 candidates, totalling 30 fits
2026-03-08 20:25:35,186 - __main__ - INFO - Mejores parámetros encontrados: {'subsample': 0.8, 'n_estimators': 50, 'min_child_weight': 500, 'max_depth': 6, 'learning_rate': 0.05, 'colsample_bytree': 1.0}
2026-03-08 20:25:35,187 - __main__ - INFO - Tiempo de ejecución: 84.63 segundos
[RMSE val con Lags y RS] 2.2179

2026-03-08 20:25:51 Uploading - Uploading generated training model
2026-03-08 20

## Hosting del modeloDeploy del modelo

Deploy del modelo

### Deploy del modelo

Desplegar el modelo en SageMaker hosting solo requiere llamar a `deploy` sobre el modelo entrenado. Esta llamada recibe un instance count, instance type, y opcionalmente funciones serializer y deserializer. Estas se usan cuando se crea el predictor resultante en el endpoint.

In [13]:
import sagemaker
sagemaker.__version__

'2.245.0'

In [22]:
from sagemaker.serializers import CSVSerializer
predictor = tree.deploy(1, "ml.m4.xlarge", serializer=CSVSerializer())

INFO:sagemaker:Creating model with name: supermarket-2026-03-08-21-25-05-456
INFO:sagemaker:Creating endpoint-config with name supermarket-2026-03-08-21-25-05-456
INFO:sagemaker:Creating endpoint with name supermarket-2026-03-08-21-25-05-456


----!

### Seleccionar datos y usarlos para una predicción

Para hacer algunas predicciones, extraeremos parte de los datos que usamos para training y haremos predicciones sobre ellos. Esto es, por supuesto, mala práctica estadística, pero es una buena forma de ver cómo funciona el mecanismo.

In [37]:
import pandas as pd
import io

datos_prueba = pd.DataFrame({
    'date_block_num': [20, 20],         # mes
    'shop_id': [40, 40],                # tienda
    'item_id': [4481, 3947],            # producto
    'item_cnt_month_lag_1': [3.0, 0.0], # ventas hace 1 mes
    'item_cnt_month_lag_2': [1.0, 2.0], # ventas hace 2 meses
    'item_cnt_month_lag_3': [0.0, 1.0]  # ventas hace 3 meses
})


In [38]:
payload_csv = datos_prueba.to_csv(header=False, index=False)
respuesta_cruda = predictor.predict(payload_csv)
texto_respuesta = respuesta_cruda.decode('utf-8')
predicciones = [float(valor) for valor in texto_respuesta.strip().split('\n')]
datos_prueba['prediccion_ventas'] = predicciones
print(datos_prueba[['shop_id', 'item_id', 'prediccion_ventas']])

   shop_id  item_id  prediccion_ventas
0       40     4481           2.221650
1       40     3947           1.629031


### Limpieza opcional
Cuando hayas terminado con el endpoint, querrás eliminarlo.

In [20]:
sess.delete_endpoint(predictor.endpoint)

See: https://sagemaker.readthedocs.io/en/stable/v2.html for details.
INFO:sagemaker:Deleting endpoint with name: supermarket-2026-03-08-07-21-04-415


### Crear un Transform Job
Crearemos un `Transformer` que define cómo usar el container para obtener resultados de inferencia sobre un dataset. Esto incluye la configuración necesaria para invocar el batch transform de SageMaker:

* El __instance count__ que es el número de máquinas para extraer inferencias
* El __instance type__ que es el tipo de máquina para extraer inferencias
* El __output path__ determina dónde se escribirán los resultados de inferencia

In [ ]:
transform_output_folder = "batch-transform-output"

# If a default bucket prefix is specified, append it to the s3 path
if default_bucket_prefix:
    transform_output_folder = f"{default_bucket_prefix}/{transform_output_folder}"

output_path = "s3://{}/{}".format(default_bucket, transform_output_folder)

transformer = tree.transformer(
    instance_count=1,
    instance_type="ml.m4.xlarge",
    output_path=output_path,
    assemble_with="Line",
    accept="text/csv",
)

Usamos transform() en el transformer para obtener resultados de inferencia sobre los datos que subimos. Puedes usar estas opciones al invocar el transformer.

* La __data_location__ es la ubicación de los datos de input
* El __content_type__ es el content type que se establece al hacer el HTTP request al container para obtener la predicción
* El __split_type__ es el delimitador usado para dividir los datos de input
* El __input_filter__ indica que la primera columna (ID) del input se eliminará antes de hacer el HTTP request al container

In [ ]:
transformer.transform(
    data_location, content_type="text/csv", split_type="Line", input_filter="$[1:]"
)
transformer.wait()

Para más información sobre las opciones de configuración, ver [CreateTransformJob API](https://docs.aws.amazon.com/sagemaker/latest/dg/API_CreateTransformJob.html)

### Ver el output
Leamos los resultados del batch transform job anterior desde los archivos en S3 e imprimamos el output

In [ ]:
s3_client = sess.boto_session.client("s3")
s3_client.download_file(
    default_bucket, "{}/iris.csv.out".format(transform_output_folder), "/tmp/iris.csv.out"
)
with open("/tmp/iris.csv.out") as f:
    results = f.readlines()
print("Transform results: \n{}".format("".join(results)))